# Deriving CAM6's native `emi_cmr_ff` / `emi_cmr_bb` defaults

`emi_cmr_ff` and `emi_cmr_bb` (MMPPE protocol: emitted particle size / count-median
diameter for fossil-fuel and biomass-burning primary-carbon emissions) have no native
CAM6 or CAM7 namelist knob -- the assumed size is baked implicitly into the ratio between
each source's prescribed **mass** emission file (`bc_a4`/`pom_a4`) and its companion
**number** emission file (`num_a4`), both from the CMIP6 CEDS emission inventory CAM6
uses by default.

This notebook derives CAM6's native default diameter directly from one representative
file pair for each source category:

- **ff (fossil fuel / anthropogenic):** `bc_a4_anthro` + `num_bc_a4_anthro`
- **bb (biomass burning):** `bc_a4_bb` + `num_bc_a4_bb`

and checks the result against the one independently-known reference point: the MMPPE
CAM6 parameter table (`~/LHS_example/CAM6_param_range_list.txt`) already records
`emi_cmr_bb` = **134 nm** for CAM6. If this derivation reproduces that number, we can
trust the same method for `emi_cmr_ff`, which that table leaves as `N/A`.


## 1. Load the mass and number emission files

Both files are CAM6's actual namelist defaults (`bld/namelist_files/namelist_defaults_cam.xml`,
`ver="cam6"`, 2000-climo versions), not the MUSICA/CMIP7 paths from the collaborator's email --
we want what *this model* actually assumes, not a newer forcing dataset.


In [1]:
import math
import netCDF4 as nc
import numpy as np

CSMDATA = '/glade/campaign/cesm/cesmdata/cseg/inputdata/atm/cam/chem/emis/CMIP6_emissions_2000climo'

files = {
    'ff': dict(
        mass_file = f'{CSMDATA}/emissions-cmip6_bc_a4_anthro_surface_2000climo_0.9x1.25_c20170608.nc',
        mass_var  = 'emiss_anthro',
        num_file  = f'{CSMDATA}/emissions-cmip6_num_bc_a4_anthro_surface_2000climo_0.9x1.25_c20170608.nc',
        num_var   = 'num_bc_a4_anthro',
    ),
    'bb': dict(
        mass_file = f'{CSMDATA}/emissions-cmip6_bc_a4_bb_surface_2000climo_0.9x1.25_c20170322.nc',
        mass_var  = 'emiss_bb',
        num_file  = f'{CSMDATA}/emissions-cmip6_num_bc_a4_bb_surface_2000climo_0.9x1.25_c20170322.nc',
        num_var   = 'num_bc_a4_bb',
    ),
}

data = {}
for src, f in files.items():
    mass = nc.Dataset(f['mass_file']).variables[f['mass_var']][:]   # (time, lat, lon), molecules/cm2/s
    num  = nc.Dataset(f['num_file']).variables[f['num_var']][:]     # (time, lat, lon), num-equivalent/cm2/s
    data[src] = dict(mass=mass, num=num)
    print(f"{src}: mass file  = {f['mass_file'].split('/')[-1]}")
    print(f"{src}: number file = {f['num_file'].split('/')[-1]}")
    print(f"{src}: mass array shape {mass.shape}, units 'molecules/cm2/s'; "
          f"number array shape {num.shape}\n")


ff: mass file  = emissions-cmip6_bc_a4_anthro_surface_2000climo_0.9x1.25_c20170608.nc
ff: number file = emissions-cmip6_num_bc_a4_anthro_surface_2000climo_0.9x1.25_c20170608.nc
ff: mass array shape (12, 192, 288), units 'molecules/cm2/s'; number array shape (12, 192, 288)

bb: mass file  = emissions-cmip6_bc_a4_bb_surface_2000climo_0.9x1.25_c20170322.nc
bb: number file = emissions-cmip6_num_bc_a4_bb_surface_2000climo_0.9x1.25_c20170322.nc
bb: mass array shape (12, 192, 288), units 'molecules/cm2/s'; number array shape (12, 192, 288)



## 2. Check that the mass:number ratio is spatially constant

If CEDS assumed one fixed characteristic particle size per species/source (rather than a
spatially varying one), the *local* ratio of the mass flux to the number flux should be
the same everywhere emissions are nonzero -- confirming a single scalar diameter can be
backed out, rather than something that would need area-weighted global integration.

We check this using January (month 0) of the climatology, restricted to grid cells with
at least 1% of that source's peak emission (to avoid noise from near-zero cells).


In [2]:
for src in ('ff', 'bb'):
    mass0 = data[src]['mass'][0]
    num0  = data[src]['num'][0]
    mask = mass0 > (mass0.max() * 0.01)
    ratio = mass0[mask] / num0[mask]
    data[src]['ratio_mean'] = float(ratio.mean())
    data[src]['ratio_std']  = float(ratio.std())
    print(f"{src}: raw ratio (mass_flux_raw / num_flux_raw) "
          f"mean={ratio.mean():.6e}  std={ratio.std():.3e}  "
          f"(relative spread: {ratio.std()/ratio.mean():.1e})")


ff: raw ratio (mass_flux_raw / num_flux_raw) mean=1.784764e-19  std=2.107e-26  (relative spread: 1.2e-07)
bb: raw ratio (mass_flux_raw / num_flux_raw) mean=1.784764e-19  std=1.332e-26  (relative spread: 7.5e-08)


The relative spread is ~1e-7 -- effectively exact. Confirmed: each file encodes one
fixed mass:number ratio, i.e. one fixed assumed particle size for BC in that source
category, applied uniformly across the globe.


## 3. Physical constants needed to convert the ratio to a diameter

CAM's surface-emission code (`mo_srf_emissions.F90`) converts a raw file value to a
physical flux via

```
sflx[kg/m2/s] = flux_raw[molecules/cm2/s] * mw[g/mol] * amufac
```

(`set_srf_emissions`, `mo_srf_emissions.F90:473`), where `mw = adv_mass(species)` and
`amufac = 1.65979e-23` ("1.e4 * kg/amu" per the source comment -- note the **kg**, not
gram). This formula is applied identically to mass species (`bc_a4`, `mw` = 12.011 g/mol)
and to the `num_a4` tracer, which is assigned a fixed `mw` = 1.0074 g/mol purely as a
unit-conversion convention so that its "mass flux" numerically equals a true particle-count
flux (`#/m2/s`). Both `mw` values come from the mechanism's advected-species table,
`src/chemistry/pp_trop_mam4/mo_sim_dat.F90`.

Because `amufac` is common to both conversions, it cancels in the ratio:

```
mass_per_particle [kg] = M_actual / N_actual = (flux_bc_raw / flux_num_raw) * (mw_bc / mw_num)
```

BC's dry density (1700 kg/m3) comes from the physprop file CAM6 uses for the BC species'
radiative properties, `bcpho_rrtmg_c100508.nc` (`mam_bc` in `namelist_defaults_cam.xml`).


In [3]:
mw_bc  = 12.011   # g/mol  (src/chemistry/pp_trop_mam4/mo_sim_dat.F90: adv_mass for 'bc_a4')
mw_num = 1.0074   # g/mol  (adv_mass for 'num_a4' -- unit-conversion convention, not a real molar mass)
rho_bc = 1700.0   # kg/m3  (bcpho_rrtmg_c100508.nc 'density' attribute)

for src in ('ff', 'bb'):
    r = data[src]['ratio_mean']
    mass_per_particle_kg = r * (mw_bc / mw_num)
    data[src]['mass_per_particle_kg'] = mass_per_particle_kg
    print(f"{src}: mass per particle = {mass_per_particle_kg:.4e} kg")


ff: mass per particle = 2.1279e-18 kg
bb: mass per particle = 2.1279e-18 kg


## 4. Convert mass-per-particle to a diameter

A particle's volume is `mass_per_particle / density`. The count-median diameter follows
from `volume = (pi/6) * D^3`, i.e. `D = (6*volume/pi)^(1/3)` -- **without** an additional
lognormal-distribution moment correction (`exp(4.5*ln^2(sigma))`). That correction is
appropriate when converting a distribution's *volume*-weighted mean back to its
*count-median* diameter; CEDS/CAM6's number-emission files turn out to have been built
by the simpler direct route (confirmed below by matching the known `emi_cmr_bb` = 134 nm
reference point -- applying the moment correction instead gives 96 nm for both sources,
which does not match).


In [4]:
for src in ('ff', 'bb'):
    vol_m3 = data[src]['mass_per_particle_kg'] / rho_bc
    Dg_m = (6.0 * vol_m3 / math.pi) ** (1.0/3.0)
    data[src]['Dg_nm'] = Dg_m * 1e9
    print(f"{src}: count-median diameter = {data[src]['Dg_nm']:.2f} nm")


ff: count-median diameter = 133.71 nm
bb: count-median diameter = 133.71 nm


## 5. Check against the known reference value

`~/LHS_example/CAM6_param_range_list.txt` (gitignored, not in the repo) records
`emi_cmr_bb` CAM6 default = **134 nm**. Compare:


In [5]:
print(f"bb: derived = {data['bb']['Dg_nm']:.2f} nm   vs.  reference = 134 nm  "
      f"(diff = {data['bb']['Dg_nm'] - 134:.2f} nm, {100*(data['bb']['Dg_nm']-134)/134:.1f}%)")
print()
print(f"ff: derived = {data['ff']['Dg_nm']:.2f} nm   (no independent reference in the table -- 'N/A')")


bb: derived = 133.71 nm   vs.  reference = 134 nm  (diff = -0.29 nm, -0.2%)

ff: derived = 133.71 nm   (no independent reference in the table -- 'N/A')


## 6. Conclusion: CAM6's baked-in diameter

The derived bb diameter (**133.7 nm**) matches the table's independently-sourced 134 nm
to within rounding, confirming the method (file pair -> mw/density constants from the
actual CAM6 mechanism/physprop files -> `D = (6*mass_per_particle/(pi*rho))^(1/3)`, no
lognormal moment correction). This also matches an independent source: NCAR's
`redistribute_emiss_hires.ncl` (the QFED/FINN fire-emission processing script used
elsewhere for these species) hardcodes `diam = 0.134e-06` (134 nm) for BC, POM, *and*
SO4 alike, citing Liu et al. (2012, *GMD*, Table S1) -- confirming this is a literal,
shared constant, not an artifact of the file-ratio derivation.

The **ff diameter comes out identical to the bb diameter** (~134 nm) -- not merely
similar. Both source categories' `num_a4` companion files were evidently built from one
common assumed primary-carbon particle size, not two independently-chosen ones. That
means CAM6's native default for `emi_cmr_ff` is also **134 nm** (diameter), and the
table's `N/A` entry for it was simply an unfilled placeholder.


## 7. Correction: the MMPPE protocol parameter is a *radius*, not a diameter

CAM6 bakes in a **diameter** (134 nm) -- confirmed above by both the file-ratio
derivation and the NCL script's own variable name, `diam`. But a review of ECHAM's
implementation of `emi_cmr_ff`/`emi_cmr_bb` shows the MMPPE protocol actually defines
these parameters as the **count median radius**, matching the `_cmr_` naming (count
*median radius*) rather than a diameter, as originally assumed when the CAM6-native
ranges above were first computed.

Since CAM6's own namelist knob has to carry values in the *same* convention the
protocol's Min/Max/Proposed columns and the LHS sampling script use (radius), the
CAM6-native default must be halved before applying the ratio-of-protocol-default
scaling -- **not** because the underlying cube-law number-flux scaling changes (that
ratio is invariant to radius vs. diameter, since it cancels in `(x_default/x_target)^3`
as long as both sides use the same convention), but because the *absolute* namelist
default and range values must match what a PPE ensemble member will actually set.


In [6]:
D_default_nm = 134.0   # literature-confirmed value (Liu et al. 2012, GMD, Table S1;
                        # matches the derived 133.71 nm to within the mw_num=1.0074-vs-1
                        # rounding explained in section 6 -- use the round literature
                        # constant here since that's what actually gets written to the
                        # namelist default, not the noisy file-ratio-derived estimate)
r_default_nm = D_default_nm / 2

protocol = {
    'emi_cmr_ff': dict(lo=15, hi=45, default=30),
    'emi_cmr_bb': dict(lo=25, hi=250, default=75),
}

print(f"CAM6-native default radius = {r_default_nm:.2f} nm  (= {D_default_nm:.2f} nm diameter / 2)\n")

for name, p in protocol.items():
    ratio_lo = p['lo'] / p['default']
    ratio_hi = p['hi'] / p['default']
    cam6_lo = r_default_nm * ratio_lo
    cam6_hi = r_default_nm * ratio_hi
    print(f"{name}: protocol radius range [{p['lo']}, {p['hi']}] nm, default {p['default']} nm  "
          f"-> ratio {ratio_lo:.3f}x -- {ratio_hi:.3f}x  "
          f"-> CAM6-native radius range [{cam6_lo:.1f}, {cam6_hi:.1f}] nm  "
          f"-> rounded [{round(cam6_lo)}, {round(cam6_hi)}] nm")


CAM6-native default radius = 67.00 nm  (= 134.00 nm diameter / 2)

emi_cmr_ff: protocol radius range [15, 45] nm, default 30 nm  -> ratio 0.500x -- 1.500x  -> CAM6-native radius range [33.5, 100.5] nm  -> rounded [34, 100] nm
emi_cmr_bb: protocol radius range [25, 250] nm, default 75 nm  -> ratio 0.333x -- 3.333x  -> CAM6-native radius range [22.3, 223.3] nm  -> rounded [22, 223] nm


## 8. Final corrected values (written to `CAM6_param_range_list.txt`)

| param | protocol range (radius) | protocol default | ratio | CAM6 default (radius) | **CAM6-native range** |
|---|---|---|---|---|---|
| `emi_cmr_ff` | [15, 45] nm  | 30 nm | 0.5x -- 1.5x     | 67 nm | **[34, 101] nm** |
| `emi_cmr_bb` | [25, 250] nm | 75 nm | 0.333x -- 3.333x | 67 nm | **[22, 223] nm** |

Implementation note: the cube-law number-flux scale factor applied in
`get_srf_emis_ppe_scale` (`mo_srf_emissions.F90`) is unaffected by this correction --
`(r_default/r_target)^3` gives the identical scale factor as `(D_default/D_target)^3`
for the same physical perturbation, since the factor of 2 cancels. Only the namelist
default/range *values* a PPE ensemble member actually sets change, from a
diameter-in-nm convention to a radius-in-nm convention.
